# Plant Biodiversity Data Analysis
## Pandas, Matplotlib and Seaborn Teaching Notebook

**Dataset:** `draft list.xlsx`

This notebook uses the real Excel sheet provided for the class. The data contains plant counts from **five transects**, measured during **pre-monsoon** and **post-monsoon** periods.

The spreadsheet is not analysis-ready. That is useful for teaching because students get to work through the same stages used in a real data project:

**Messy Excel → Pandas → Cleaning → Tidy Data → Analysis → Matplotlib → Seaborn → Interpretation**

### Learning outcomes

By the end of this notebook, students should be able to:

- load an Excel file with Pandas
- inspect rows, columns, types and missing values
- select, filter and sort data
- clean text and numeric columns
- reshape a messy spreadsheet into tidy data
- use `groupby()`, `agg()`, `value_counts()`, `pivot_table()` and `melt()`
- calculate ecological summaries such as abundance and species richness
- create charts with Matplotlib
- create statistical visualizations with Seaborn
- compare pre-monsoon and post-monsoon plant abundance
- check whether a data-cleaning pipeline preserved the original totals

## 0. About this Excel file

The original worksheet stores five transects side by side.

Each transect uses three columns:

| Column | Meaning |
|---|---|
| Plant | Plant or species name |
| Pre-monsoon | Number of individuals before monsoon |
| Post-monsoon | Number of individuals after monsoon |

The original file also contains:

- multiple header rows
- blank cells
- inconsistent capitalization
- extra spaces in plant names
- a `TOTAL` row
- one malformed header/data row in the Transect 4 block

This means simply running `pd.read_excel()` is only the beginning. We need to understand the structure before analysis.

## 1. Setup

If you are running this notebook locally and the packages are missing, install them with:

```bash
pip install pandas matplotlib seaborn openpyxl
```

`openpyxl` is used by Pandas to read `.xlsx` files.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

### Find the dataset

Keep `draft list.xlsx` in the same folder as this notebook.

The fallback `/mnt/data/` path is included so the notebook also works inside ChatGPT's file environment.

In [ ]:
possible_paths = [
    Path("draft list.xlsx"),
    Path("/mnt/data/draft list.xlsx"),
]

DATA_PATH = next((path for path in possible_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find 'draft list.xlsx'. "
        "Place the Excel file in the same folder as this notebook."
    )

print("Using dataset:", DATA_PATH.resolve())

# Part A. Pandas

## 2. Load the raw Excel sheet

We deliberately use `header=None`.

Why?

Because this worksheet has several header rows, and asking Pandas to automatically choose a header would hide part of the spreadsheet structure that we need to clean.

In [ ]:
raw = pd.read_excel(
    DATA_PATH,
    sheet_name="Sheet1",
    header=None
)

raw.head(10)

### First inspection

Never begin cleaning immediately. First ask:

- How many rows and columns are there?
- What does the first row contain?
- Where are the headings?
- Where does the actual data begin?
- Is there a total row?

In [ ]:
print("Shape:", raw.shape)
print("\nFirst row:")
print(raw.iloc[0].tolist())

print("\nLast five rows:")
display(raw.tail())

### Exercise 1

Without changing the dataframe, answer:

1. How many rows are present?
2. How many columns are present?
3. How many transects can you identify?
4. What pattern do the columns follow?
5. Where is the `TOTAL` row located?

Try using:

```python
raw.shape
raw.iloc[]
raw.head()
raw.tail()
```

In [ ]:
# STUDENT EXERCISE 1
# Write your exploration code below.

## 3. Understand the column blocks

Each transect occupies three columns.

The starting positions are:

- Transect 1: column 0
- Transect 2: column 3
- Transect 3: column 6
- Transect 4: column 9
- Transect 5: column 12

In [ ]:
BLOCK_STARTS = [0, 3, 6, 9, 12]

for transect_number, start in enumerate(BLOCK_STARTS, start=1):
    print(f"Transect {transect_number}")
    display(raw.iloc[:8, start:start+3])

### Data-quality observation

Notice that **Transect 4** is slightly different from the other blocks.

`Ailanthes sp.` appears in the row where the pre/post labels are placed, but there are no numeric counts attached to that entry. We should not invent values for it.

A good cleaning pipeline should:

1. keep valid numeric records,
2. discard header text,
3. preserve genuine zero counts,
4. avoid inventing missing values.

## 4. Reshape the spreadsheet into a clean table

Our target structure is:

| Transect | Plant | Pre_Monsoon | Post_Monsoon |
|---|---|---:|---:|
| Transect 1 | Adiantum latifolium | 2 | 2 |
| Transect 1 | Agrostis capillaris | 2 | 1 |
| ... | ... | ... | ... |

This is much easier to analyze than five separate blocks.

In [ ]:
blocks = []

for transect_number, start in enumerate(BLOCK_STARTS, start=1):
    block = raw.iloc[2:-1, start:start+3].copy()

    block.columns = [
        "Plant",
        "Pre_Monsoon",
        "Post_Monsoon",
    ]

    # Convert counts to numbers.
    # Header labels such as "pre monsoon" become NaN.
    block["Pre_Monsoon"] = pd.to_numeric(
        block["Pre_Monsoon"],
        errors="coerce"
    )

    block["Post_Monsoon"] = pd.to_numeric(
        block["Post_Monsoon"],
        errors="coerce"
    )

    # Clean surrounding and repeated spaces in plant names.
    block["Plant"] = (
        block["Plant"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    # A valid record needs a plant name and at least one numeric count.
    block = block[
        block["Plant"].notna()
        & block[["Pre_Monsoon", "Post_Monsoon"]]
            .notna()
            .any(axis=1)
    ].copy()

    block.insert(
        0,
        "Transect",
        f"Transect {transect_number}"
    )

    blocks.append(block)

clean = pd.concat(blocks, ignore_index=True)

clean.head(10)

### Check the cleaned dataframe

In [ ]:
print("Shape:", clean.shape)
print("\nColumns:", clean.columns.tolist())
print("\nData types:")
print(clean.dtypes)

display(clean.head())
display(clean.tail())

## 5. Validate the cleaning

A cleaning step can run without errors and still be wrong.

The original workbook gives total counts for each transect. We can calculate our own totals and compare them with the Excel totals.

This is a simple but powerful validation habit.

In [ ]:
computed_totals = (
    clean
    .groupby("Transect")[["Pre_Monsoon", "Post_Monsoon"]]
    .sum()
)

computed_totals

In [ ]:
declared_totals = pd.DataFrame(
    {
        "Declared_Pre": [
            raw.iloc[-1, 1],
            raw.iloc[-1, 4],
            raw.iloc[-1, 7],
            raw.iloc[-1, 10],
            raw.iloc[-1, 13],
        ],
        "Declared_Post": [
            raw.iloc[-1, 2],
            raw.iloc[-1, 5],
            raw.iloc[-1, 8],
            raw.iloc[-1, 11],
            raw.iloc[-1, 14],
        ],
    },
    index=[
        "Transect 1",
        "Transect 2",
        "Transect 3",
        "Transect 4",
        "Transect 5",
    ],
)

validation = computed_totals.join(declared_totals)

validation["Pre_Matches"] = (
    validation["Pre_Monsoon"]
    == validation["Declared_Pre"]
)

validation["Post_Matches"] = (
    validation["Post_Monsoon"]
    == validation["Declared_Post"]
)

validation

In [ ]:
assert validation["Pre_Matches"].all()
assert validation["Post_Matches"].all()

print("Validation passed. All computed totals match the Excel totals.")

### Trainer sanity-check

The expected totals are:

| Transect | Pre | Post |
|---|---:|---:|
| Transect 1 | 93 | 91 |
| Transect 2 | 86 | 96 |
| Transect 3 | 88 | 82 |
| Transect 4 | 72 | 60 |
| Transect 5 | 93 | 88 |

If the student's cleaning produces different totals, inspect the filtering and numeric conversion steps.

## 6. Core Pandas inspection methods

In [ ]:
clean.head()

In [ ]:
clean.tail()

In [ ]:
clean.sample(5, random_state=42)

In [ ]:
clean.info()

In [ ]:
clean.describe()

### What does `describe()` tell us?

For numeric columns, it gives:

- `count`
- `mean`
- `std`
- `min`
- quartiles
- `max`

It is a fast way to spot unusual values.

## 7. Selecting columns

In [ ]:
clean["Plant"].head()

In [ ]:
clean[["Transect", "Plant", "Pre_Monsoon"]].head()

## 8. Selecting rows with `loc` and `iloc`

`iloc` selects by integer position.

`loc` selects using labels and conditions.

In [ ]:
clean.iloc[0]

In [ ]:
clean.iloc[0:5, 0:3]

In [ ]:
clean.loc[0:5, ["Transect", "Plant", "Post_Monsoon"]]

## 9. Filtering data

Find species records with more than five post-monsoon individuals.

In [ ]:
clean[clean["Post_Monsoon"] > 5]

### Multiple conditions

Find records in Transect 5 where post-monsoon count is greater than pre-monsoon count.

In [ ]:
clean[
    (clean["Transect"] == "Transect 5")
    & (clean["Post_Monsoon"] > clean["Pre_Monsoon"])
]

### The same idea using `query()`

In [ ]:
clean.query(
    "Transect == 'Transect 5' and Post_Monsoon > Pre_Monsoon"
)

## 10. Sorting

In [ ]:
clean.sort_values(
    "Post_Monsoon",
    ascending=False
).head(10)

In [ ]:
clean.sort_values(
    ["Transect", "Pre_Monsoon"],
    ascending=[True, False]
).head(15)

## 11. Missing values and duplicates

In [ ]:
clean.isna().sum()

In [ ]:
clean.duplicated().sum()

Repeated plant names across different transects are **not** necessarily duplicate rows.

A more meaningful duplicate check is:

In [ ]:
clean.duplicated(
    subset=["Transect", "Plant"],
    keep=False
).sum()

## 12. Text cleaning

Plant names contain differences such as:

- leading or trailing spaces
- repeated spaces
- different capitalization

We should be careful with scientific names. Automatically correcting spelling can create biological errors.

For analysis, we can create a normalized key while preserving the original cleaned name.

In [ ]:
clean["Plant_Key"] = (
    clean["Plant"]
    .str.casefold()
    .str.strip()
)

clean[["Plant", "Plant_Key"]].head(15)

### Inspect possible naming inconsistencies

This does not automatically decide that two spellings are the same species. It simply helps us find entries that deserve review.

In [ ]:
sorted(clean["Plant"].dropna().unique())[:50]

## 13. Create a tidy, long-format dataframe

Seaborn works especially well with tidy data.

Instead of:

| Plant | Pre_Monsoon | Post_Monsoon |
|---|---:|---:|
| Species A | 2 | 4 |

we want:

| Plant | Season | Individuals |
|---|---|---:|
| Species A | Pre-monsoon | 2 |
| Species A | Post-monsoon | 4 |

In [ ]:
tidy = clean.melt(
    id_vars=["Transect", "Plant", "Plant_Key"],
    value_vars=["Pre_Monsoon", "Post_Monsoon"],
    var_name="Season",
    value_name="Individuals",
)

tidy["Season"] = tidy["Season"].replace(
    {
        "Pre_Monsoon": "Pre-monsoon",
        "Post_Monsoon": "Post-monsoon",
    }
)

tidy = tidy.dropna(subset=["Individuals"]).reset_index(drop=True)

tidy.head(10)

### Why tidy data?

A common rule is:

- one variable per column
- one observation per row
- one type of observational unit per table

Here:

- `Transect` is a variable
- `Plant` is a variable
- `Season` is a variable
- `Individuals` is a variable

## 14. `value_counts()`

How many plant records are present in each transect?

In [ ]:
clean["Transect"].value_counts()

## 15. `groupby()`

### Total abundance by transect

In [ ]:
clean.groupby("Transect")[["Pre_Monsoon", "Post_Monsoon"]].sum()

### Total abundance by transect and season

In [ ]:
abundance_summary = (
    tidy
    .groupby(["Transect", "Season"], as_index=False)
    ["Individuals"]
    .sum()
)

abundance_summary

### Species richness

A simple richness measure is the number of unique plant names recorded.

Here we calculate richness by transect.

In [ ]:
richness_by_transect = (
    clean
    .groupby("Transect")["Plant_Key"]
    .nunique()
    .sort_values(ascending=False)
)

richness_by_transect

### Richness by transect and season

A plant is counted as present in a season only when its count is greater than zero.

In [ ]:
season_presence = tidy[tidy["Individuals"] > 0]

richness_by_season = (
    season_presence
    .groupby(["Transect", "Season"])["Plant_Key"]
    .nunique()
    .unstack(fill_value=0)
)

richness_by_season

## 16. `agg()`

Calculate several statistics at the same time.

In [ ]:
clean.groupby("Transect").agg(
    Species_Records=("Plant", "count"),
    Mean_Pre=("Pre_Monsoon", "mean"),
    Mean_Post=("Post_Monsoon", "mean"),
    Max_Pre=("Pre_Monsoon", "max"),
    Max_Post=("Post_Monsoon", "max"),
)

## 17. Which plants are most abundant?

First aggregate a plant across all transects.

In [ ]:
plant_abundance = (
    clean
    .groupby(["Plant_Key"], as_index=False)
    .agg(
        Plant=("Plant", "first"),
        Pre_Monsoon=("Pre_Monsoon", "sum"),
        Post_Monsoon=("Post_Monsoon", "sum"),
    )
)

plant_abundance["Total"] = (
    plant_abundance["Pre_Monsoon"]
    + plant_abundance["Post_Monsoon"]
)

plant_abundance.sort_values(
    "Total",
    ascending=False
).head(10)

## 18. Calculate change from pre-monsoon to post-monsoon

In [ ]:
clean["Change"] = (
    clean["Post_Monsoon"]
    - clean["Pre_Monsoon"]
)

clean["Percent_Change"] = (
    clean["Change"]
    .div(clean["Pre_Monsoon"].replace(0, pd.NA))
    .mul(100)
)

clean[
    [
        "Transect",
        "Plant",
        "Pre_Monsoon",
        "Post_Monsoon",
        "Change",
        "Percent_Change",
    ]
].head(10)

### Largest increases

In [ ]:
clean.sort_values(
    "Change",
    ascending=False
).head(10)

### Largest decreases

In [ ]:
clean.sort_values(
    "Change",
    ascending=True
).head(10)

## 19. `pivot_table()`

Create a transect-by-season summary.

In [ ]:
transect_pivot = pd.pivot_table(
    tidy,
    index="Transect",
    columns="Season",
    values="Individuals",
    aggfunc="sum",
    fill_value=0,
)

transect_pivot

# Part B. Matplotlib

## 20. Matplotlib basics

Matplotlib gives us direct control over a figure.

The basic pattern is:

```python
plt.figure(...)
plt.bar(...)       # or plt.plot(), plt.scatter(), plt.hist()
plt.xlabel(...)
plt.ylabel(...)
plt.title(...)
plt.show()
```

## 21. Bar chart: total abundance by transect

We will create grouped bars for pre-monsoon and post-monsoon totals.

In [ ]:
summary = (
    clean
    .groupby("Transect")[["Pre_Monsoon", "Post_Monsoon"]]
    .sum()
)

x = list(range(len(summary)))
width = 0.38

plt.figure(figsize=(10, 5))

plt.bar(
    [i - width / 2 for i in x],
    summary["Pre_Monsoon"],
    width=width,
    label="Pre-monsoon",
)

plt.bar(
    [i + width / 2 for i in x],
    summary["Post_Monsoon"],
    width=width,
    label="Post-monsoon",
)

plt.xticks(x, summary.index)
plt.xlabel("Transect")
plt.ylabel("Total individuals")
plt.title("Plant Abundance Before and After Monsoon")
plt.legend()
plt.tight_layout()
plt.show()

### Reading the chart

Ask students:

- Which transect has the highest pre-monsoon abundance?
- Which has the highest post-monsoon abundance?
- Which transects increased?
- Which transects decreased?

## 22. Line plot: abundance across transects

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    summary.index,
    summary["Pre_Monsoon"],
    marker="o",
    label="Pre-monsoon",
)

plt.plot(
    summary.index,
    summary["Post_Monsoon"],
    marker="o",
    label="Post-monsoon",
)

plt.xlabel("Transect")
plt.ylabel("Total individuals")
plt.title("Abundance Across Transects")
plt.legend()
plt.tight_layout()
plt.show()

## 23. Histogram: distribution of individual counts

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    tidy["Individuals"],
    bins=10,
    edgecolor="black",
)

plt.xlabel("Individuals per plant record")
plt.ylabel("Frequency")
plt.title("Distribution of Plant Counts")
plt.tight_layout()
plt.show()

## 24. Scatter plot: pre vs post abundance

First aggregate each plant across all transects.

In [ ]:
plant_pre_post = (
    clean
    .groupby("Plant_Key", as_index=False)
    .agg(
        Plant=("Plant", "first"),
        Pre=("Pre_Monsoon", "sum"),
        Post=("Post_Monsoon", "sum"),
    )
)

plt.figure(figsize=(7, 6))

plt.scatter(
    plant_pre_post["Pre"],
    plant_pre_post["Post"],
    alpha=0.7,
)

limit = max(
    plant_pre_post["Pre"].max(),
    plant_pre_post["Post"].max(),
)

plt.plot([0, limit], [0, limit], linestyle="--")

plt.xlabel("Pre-monsoon individuals")
plt.ylabel("Post-monsoon individuals")
plt.title("Plant Abundance: Pre vs Post Monsoon")
plt.tight_layout()
plt.show()

### How to interpret the diagonal

- points above the line increased after monsoon
- points below the line decreased
- points near the line changed little

# Part C. Seaborn

## 25. Why use Seaborn?

Seaborn is built on Matplotlib and works naturally with Pandas dataframes.

It is especially useful when the chart depends on categorical variables such as:

- transect
- season
- species group

The general pattern is:

```python
sns.some_plot(
    data=dataframe,
    x="column",
    y="column",
    hue="another_column"
)
```

## 26. Seaborn bar plot

The raw records can be aggregated directly inside the plot.

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=tidy,
    x="Transect",
    y="Individuals",
    hue="Season",
    estimator=sum,
    errorbar=None,
)

plt.title("Total Individuals by Transect and Season")
plt.ylabel("Total individuals")
plt.tight_layout()
plt.show()

## 27. Seaborn box plot

A box plot shows the distribution of individual plant counts rather than only the total.

In [ ]:
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=tidy,
    x="Transect",
    y="Individuals",
    hue="Season",
)

plt.title("Distribution of Plant Counts by Transect")
plt.tight_layout()
plt.show()

### Discuss

A transect can have a high total abundance because:

- it contains many species,
- some species have large counts,
- or both.

The bar chart and box plot answer different questions.

## 28. Seaborn histogram

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=tidy,
    x="Individuals",
    hue="Season",
    bins=10,
    element="step",
)

plt.title("Distribution of Counts by Season")
plt.tight_layout()
plt.show()

## 29. Seaborn scatter plot

In [ ]:
plt.figure(figsize=(7, 6))

sns.scatterplot(
    data=plant_pre_post,
    x="Pre",
    y="Post",
    s=70,
)

limit = max(
    plant_pre_post["Pre"].max(),
    plant_pre_post["Post"].max(),
)

plt.plot([0, limit], [0, limit], linestyle="--")

plt.title("Plant-Level Pre vs Post Monsoon Abundance")
plt.tight_layout()
plt.show()

## 30. Heatmap

A heatmap is useful when we want to compare many plant-transect combinations.

To keep it readable, we will use the 15 most abundant plants.

In [ ]:
top_15_keys = (
    plant_abundance
    .sort_values("Total", ascending=False)
    .head(15)["Plant_Key"]
)

heatmap_data = (
    clean[clean["Plant_Key"].isin(top_15_keys)]
    .assign(
        Total=lambda df:
            df["Pre_Monsoon"] + df["Post_Monsoon"]
    )
    .pivot_table(
        index="Plant",
        columns="Transect",
        values="Total",
        aggfunc="sum",
        fill_value=0,
    )
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".0f",
)

plt.title("Top 15 Plants Across Transects")
plt.tight_layout()
plt.show()

# Part D. Guided Student Exercises

## Exercise 2: Filtering

Find all records where:

- pre-monsoon count is at least 5
- post-monsoon count is lower than pre-monsoon count

Show only:

`Transect`, `Plant`, `Pre_Monsoon`, `Post_Monsoon`

In [ ]:
# STUDENT EXERCISE 2

## Exercise 3: Transect summary

Create a dataframe containing:

- transect
- total pre-monsoon individuals
- total post-monsoon individuals
- absolute change

Sort from the largest increase to the largest decrease.

In [ ]:
# STUDENT EXERCISE 3

## Exercise 4: Species richness

Find the number of unique plant names recorded in each transect.

Sort from highest richness to lowest richness.

In [ ]:
# STUDENT EXERCISE 4

## Exercise 5: Plants that appeared after monsoon

Find records where:

- pre-monsoon count is `0`
- post-monsoon count is greater than `0`

What biological or sampling explanations could produce this pattern?

In [ ]:
# STUDENT EXERCISE 5

## Exercise 6: Plants lost after monsoon

Find records where:

- pre-monsoon count is greater than `0`
- post-monsoon count is `0`

Do not assume disappearance means extinction. Discuss possible explanations.

In [ ]:
# STUDENT EXERCISE 6

## Exercise 7: Largest changes

Find the five records with the largest:

1. increases
2. decreases

Use the `Change` column.

In [ ]:
# STUDENT EXERCISE 7

## Exercise 8: Matplotlib challenge

Create a horizontal bar chart showing the **10 most abundant plants across both seasons and all transects**.

Requirements:

- aggregate plants first
- sort the results
- plot only the top 10
- include title and axis labels

In [ ]:
# STUDENT EXERCISE 8

## Exercise 9: Seaborn challenge

Create a Seaborn visualization comparing pre-monsoon and post-monsoon abundance across transects.

Choose one:

- `barplot`
- `boxplot`
- `violinplot`

Then write two observations below the chart.

In [ ]:
# STUDENT EXERCISE 9

## Exercise 10: Data-quality challenge

Look through the unique plant names.

Find at least five examples that may represent:

- inconsistent capitalization
- extra spaces
- spelling variants
- possible taxonomic naming errors

Do **not** automatically merge them unless you have biological evidence that they refer to the same taxon.

In [ ]:
# STUDENT EXERCISE 10

# Part E. Mini Project

## Question

**How did plant abundance and observed plant richness change from pre-monsoon to post-monsoon across the five transects?**

### Student deliverables

Create a short analysis containing:

1. **Dataset overview**
   - rows and columns
   - structure of the original Excel file

2. **Cleaning**
   - explain how the five transect blocks were combined
   - explain how numeric values and text were cleaned

3. **Validation**
   - prove that computed totals match the Excel totals

4. **Pandas analysis**
   - total abundance by transect
   - species richness by transect
   - pre vs post change
   - top 10 abundant plants

5. **Visualizations**
   - one Matplotlib chart
   - two Seaborn charts
   - one heatmap

6. **Interpretation**
   - write at least five observations
   - separate data observations from biological explanations

### Important scientific habit

A chart tells you what the dataset shows.

It does not automatically tell you **why** the pattern occurred.

For example, a post-monsoon decline could be related to ecology, sampling effort, detectability, disturbance, seasonality or data-entry differences. More evidence is needed before choosing an explanation.

# Part F. Trainer Answer Key

The following cells contain possible solutions. Students should attempt the exercises before opening this section.

## Solution 2

In [ ]:
solution_2 = clean[
    (clean["Pre_Monsoon"] >= 5)
    & (clean["Post_Monsoon"] < clean["Pre_Monsoon"])
][
    ["Transect", "Plant", "Pre_Monsoon", "Post_Monsoon"]
]

solution_2

## Solution 3

In [ ]:
solution_3 = (
    clean
    .groupby("Transect", as_index=False)
    .agg(
        Pre=("Pre_Monsoon", "sum"),
        Post=("Post_Monsoon", "sum"),
    )
)

solution_3["Change"] = (
    solution_3["Post"]
    - solution_3["Pre"]
)

solution_3.sort_values(
    "Change",
    ascending=False
)

## Solution 4

In [ ]:
solution_4 = (
    clean
    .groupby("Transect")["Plant_Key"]
    .nunique()
    .sort_values(ascending=False)
)

solution_4

## Solution 5

In [ ]:
solution_5 = clean[
    (clean["Pre_Monsoon"] == 0)
    & (clean["Post_Monsoon"] > 0)
][
    ["Transect", "Plant", "Pre_Monsoon", "Post_Monsoon"]
]

solution_5

## Solution 6

In [ ]:
solution_6 = clean[
    (clean["Pre_Monsoon"] > 0)
    & (clean["Post_Monsoon"] == 0)
][
    ["Transect", "Plant", "Pre_Monsoon", "Post_Monsoon"]
]

solution_6

## Solution 7

In [ ]:
largest_increases = (
    clean
    .sort_values("Change", ascending=False)
    .head(5)
)

largest_decreases = (
    clean
    .sort_values("Change", ascending=True)
    .head(5)
)

print("Largest increases")
display(
    largest_increases[
        ["Transect", "Plant", "Pre_Monsoon", "Post_Monsoon", "Change"]
    ]
)

print("Largest decreases")
display(
    largest_decreases[
        ["Transect", "Plant", "Pre_Monsoon", "Post_Monsoon", "Change"]
    ]
)

## Solution 8

In [ ]:
top_10 = (
    plant_abundance
    .sort_values("Total", ascending=False)
    .head(10)
    .sort_values("Total")
)

plt.figure(figsize=(9, 6))

plt.barh(
    top_10["Plant"],
    top_10["Total"],
)

plt.xlabel("Total individuals")
plt.ylabel("Plant")
plt.title("Top 10 Most Abundant Plants")
plt.tight_layout()
plt.show()

## Solution 9

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=tidy,
    x="Transect",
    y="Individuals",
    hue="Season",
    estimator=sum,
    errorbar=None,
)

plt.ylabel("Total individuals")
plt.title("Pre vs Post Monsoon Abundance")
plt.tight_layout()
plt.show()

# Part G. Quick Revision Sheet

## Pandas commands to remember

```python
pd.read_excel()
df.head()
df.tail()
df.shape
df.columns
df.info()
df.describe()

df["column"]
df[["col1", "col2"]]

df.loc[]
df.iloc[]

df[df["column"] > value]
df.sort_values()

df.isna()
df.dropna()
df.duplicated()

df.groupby()
df.agg()
df.value_counts()
df.melt()
df.pivot_table()
```

## Matplotlib commands to remember

```python
plt.figure()
plt.plot()
plt.bar()
plt.barh()
plt.scatter()
plt.hist()

plt.xlabel()
plt.ylabel()
plt.title()
plt.legend()
plt.tight_layout()
plt.show()
```

## Seaborn commands to remember

```python
sns.barplot()
sns.boxplot()
sns.histplot()
sns.scatterplot()
sns.heatmap()
```

# Final Reflection

Ask the student to answer these without code:

1. Why was the original Excel file difficult to analyze directly?
2. What is the difference between wide and long data?
3. Why did we validate the cleaned totals?
4. When would you use a bar chart instead of a box plot?
5. What does a heatmap make easier to see?
6. Why should spelling variants in scientific names not be automatically merged?
7. What is one observation you can make about the pre/post monsoon data?
8. What extra information would you need before making a biological explanation for that observation?